# <font color='black'>Introduction to Machine Learning</font>

---

<img src="images/ipsa_logo.png" width="100" align="right">


> Year: **2025-2026**

> Contributors: **TO BE DECIDED**


## <font color='black'>Contents</font>

---
 
1. [Setting up the data](#Set-Up)
2. [A faire](#LinearRegression)
 * [Gradient Descent](#GD)
3. [A Faire](#LinearRegression2)
 * [Feature Scaling](#FeatureScaling)
 * [Gradient Descent](#GD2)
 * [Normal Equation](#NE)

Part 1:

## Set-Up
---
Let's first import the package we will need throughout this notebook.

In [30]:
#===================
# time wasted here: 5
# ==================

import numpy as np
import pandas as pd
import seaborn as sns
import time as t

import matplotlib.pyplot as plt 
#tell jupiter to display plots in the notebook.
%matplotlib inline 

Let's now import and visualize the data.

In [ ]:
data = pd.read_parquet("./data/archive/Combined_Flights_2018.parquet")
load_time_parquet = t.time() - t0

data.head()
before = data.shape
print(before)

(5689512, 61)


Part 2:

## Cleanup
---
Let's clean our data in this following part : 

In [32]:
# We are only interessted in flights that actually landed at the correct destination
data = data[(data['Cancelled'] == 0) & (data['Diverted'] == 0)]
data = data.drop(columns=['Cancelled', 'Diverted'])


# We're only keeping essential features :
cols_to_keep = [
    'Month', 'DayofMonth', 'DayOfWeek', 
    'Airline', 'Origin', 'Dest', 
    'CRSDepTime', 'Distance', 'DepDelay', 'ArrDelay'
]
data = data[cols_to_keep]

# Removing lines where the NaN values are presents in the target variables
data = data.dropna() # dropna = drop Not a Number (empty/missing values).


# Engineering des heures (Transformer HHMM en heures décimales toujours en base 24h)
# Exemple : 1530 devient 15.5 heures
"""data['DepHours'] = round(((data['CRSDepTime'] // 100) + (data['CRSDepTime'] %100)/60),2)""" 
# Temps tout en minutes (1202 devient 722 minutes)
data['DepHours'] = round(data['CRSDepTime'] // 100) * 60 + round((data['CRSDepTime'] % 100) / 60*60 )

data = data.drop(columns=['CRSDepTime'])


# Adding a binary feature (0,1) to indicate if there is a delay
data['IsDelay'] = (data['ArrDelay'] > 15).astype(int)

# Transforming text collums into categorys for memory optimisation
for col in ['Airline', 'Origin', 'Dest']:
    data[col] = data[col].astype('category')

after = data.shape
print(data.shape)
print(before[0]-after[0])
print(data.head(10))

(5585544, 11)
103968
    Month  DayofMonth  DayOfWeek            Airline Origin Dest  Distance  \
0       1          23          2  Endeavor Air Inc.    ABY  ATL     145.0   
1       1          24          3  Endeavor Air Inc.    ABY  ATL     145.0   
2       1          25          4  Endeavor Air Inc.    ABY  ATL     145.0   
3       1          26          5  Endeavor Air Inc.    ABY  ATL     145.0   
4       1          27          6  Endeavor Air Inc.    ABY  ATL     145.0   
6       1          29          1  Endeavor Air Inc.    ABY  ATL     145.0   
7       1          30          2  Endeavor Air Inc.    ABY  ATL     145.0   
9       1           3          3  Endeavor Air Inc.    ATL  ABY     145.0   
10      1           4          4  Endeavor Air Inc.    ATL  ABY     145.0   
11      1           5          5  Endeavor Air Inc.    ATL  ABY     145.0   

    DepDelay  ArrDelay  DepHours  IsDelay  
0       -5.0      -8.0     722.0        0  
1       -5.0      -6.0     722.0        0  

Part 3:

## EDA : 
---

In [33]:
plt.figure(figsize=(12, 6))

hourly_stats = data.groupby('DepHour')['IsDelay'].mean()
hourly_stats.plot(kind='bar', color='skyblue')

plt.title("Probabilité de retard (>15min) selon l'heure de départ")
plt.xlabel("Heure de la journée (0-23h)")
plt.ylabel("% de vols en retard")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

KeyError: 'DepHour'

<Figure size 1200x600 with 0 Axes>

LINEAR REGRESSION :